### Bart finetune

In [ ]:
! pip install transformers datasets accelerate

In [ ]:
!pip install scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

preprocessing

In [ ]:
! cd

for later: bart-larger

In [ ]:
# from transformers import AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large")

# def tokenize(example):
#     return tokenizer(example["text"], truncation=True, padding="max_length")

# dataset = dataset.map(tokenize, batched=True)
# dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
!pip install datasets

## DistilBERT multi-feature phishing detection

In [ ]:
import re

def extract_features(text):
    text_lower = text.lower()

    # ========================
    # REQUEST TYPE (IMPROVED)
    # ========================
    if re.search(r"(password|otp|login|verify|credentials)", text_lower):
        request_type = 1  # credentials
    elif re.search(r"(pay|payment|transfer|upi|refund|invoice)", text_lower):
        request_type = 2  # payment
    elif re.search(r"(details|information|update|confirm)", text_lower):
        request_type = 3  # personal info
    else:
        request_type = 0

    # ========================
    # MANIPULATION
    # ========================
    manipulation = int(
        bool(re.search(r"(urgent|immediately|now|suspend|limited time|action required)", text_lower))
    )

    # ========================
    # IMPERSONATION (FIXED)
    # ========================
    impersonation = int(
        bool(re.search(r"(bank of|customer support|official team|admin department)", text_lower))
    )

    # ========================
    # INTENT (DO NOT FORCE HERE)
    # ========================
    intent = 1 if (manipulation or request_type != 0 or impersonation) else 0

    return {
        "intent": intent,
        "manipulation": manipulation,
        "request_type": request_type,
        "impersonation": impersonation
    }

In [2]:
import torch
import torch.nn as nn
from transformers import DistilBertPreTrainedModel, DistilBertModel

class DistilBertMultiTaskClassifier(DistilBertPreTrainedModel):
    """Memory-optimized multi-head classifier"""

    def __init__(self, config):
        super().__init__(config)
        
        self.distilbert = DistilBertModel(config)
        self.dropout = nn.Dropout(config.dropout)
        
        # Multi-head classifiers
        self.intent_classifier = nn.Linear(config.hidden_size, 3)
        self.manipulation_classifier = nn.Linear(config.hidden_size, 2)
        self.request_type_classifier = nn.Linear(config.hidden_size, 4)
        self.impersonation_classifier = nn.Linear(config.hidden_size, 2)

        # 🔥 IMPORTANT: reduce memory
        self.config.use_cache = False

        # 🔥 OPTIONAL (huge memory saver)
        self.gradient_checkpointing_enable()

    def forward(
        self,
        input_ids,
        attention_mask=None,
        labels=None,
        **kwargs
    ):
        # Handle labels from Trainer
        if labels is None and 'intent' in kwargs:
            labels = {
                'intent': kwargs.pop('intent'),
                'manipulation': kwargs.pop('manipulation'),
                'request_type': kwargs.pop('request_type'),
                'impersonation': kwargs.pop('impersonation'),
            }

        # 🔥 IMPORTANT: return_dict=False reduces overhead
        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=False
        )

        # outputs[0] = last_hidden_state
        pooled_output = outputs[0][:, 0]   # CLS token (slightly faster)
        pooled_output = self.dropout(pooled_output)

        # Heads
        intent_logits = self.intent_classifier(pooled_output)
        manipulation_logits = self.manipulation_classifier(pooled_output)
        request_type_logits = self.request_type_classifier(pooled_output)
        impersonation_logits = self.impersonation_classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()

            # 🔥 Weighted loss (better learning, same memory)
            loss = (
                1.5 * loss_fn(intent_logits, labels["intent"]) +
                1.0 * loss_fn(manipulation_logits, labels["manipulation"]) +
                2.0 * loss_fn(request_type_logits, labels["request_type"]) +
                1.5 * loss_fn(impersonation_logits, labels["impersonation"])
            ) / 6.0

        return {
            "loss": loss,
            "intent": intent_logits,
            "manipulation": manipulation_logits,
            "request_type": request_type_logits,
            "impersonation": impersonation_logits,
        }

/mnt/DISK/Studies/SEM 4/Project Course 299/Phishing-detector/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import pandas as pd
import torch
import gc
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoConfig,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score

print("loading")

# ========================
# SAFE CSV LOAD (VERY IMPORTANT)
# ========================
df = pd.read_csv(
    "dataset/processed/emails_sms_combined.csv",
    engine="python",
    on_bad_lines="skip"
)

df = df[["channel", "text", "label"]]

# 🔥 FIX LABEL ISSUES
df["label"] = pd.to_numeric(df["label"], errors="coerce")
df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)

print(df.head())
print("Dataset size:", len(df))

# ========================
# CONVERT TO DATASET
# ========================
dataset = Dataset.from_pandas(df)

# ========================


def preprocess(example):
    features = extract_features(example["text"])

    # ONLY fix intent — do NOT overwrite everything
    if example["label"] == 1:
        features["intent"] = 1
    else:
        features["intent"] = 0

    return {
        "text": example["text"],
        "channel": example["channel"],
        **features
    }

dataset = dataset.map(preprocess)

# ========================
# TOKENIZER
# ========================
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

dataset.reset_format()

cols_to_keep = ["text", "channel", "intent", "manipulation", "request_type", "impersonation"]
dataset = dataset.remove_columns([col for col in dataset.column_names if col not in cols_to_keep])

def tokenize(example):
    text = f"{example['channel']}: {example['text']}"
    return tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128
    )

# ❌ REMOVE keep_in_memory=True (VERY IMPORTANT)
dataset = dataset.map(tokenize, batched=False)

# ========================
# FINAL FORMAT
# ========================
dataset = dataset.remove_columns([
    col for col in dataset.column_names 
    if col not in ["input_ids", "attention_mask", 
                   "intent", "manipulation", 
                   "request_type", "impersonation"]
])

# ========================
# SPLIT
# ========================
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
val_dataset = dataset["test"]

# ========================
# CLEAN GPU BEFORE MODEL LOAD
# ========================
gc.collect()
torch.cuda.empty_cache()

# ========================
# LOAD MODEL (DO NOT MOVE TO DEVICE)
# ========================
config = AutoConfig.from_pretrained("distilbert-base-uncased")
model = DistilBertMultiTaskClassifier(config)

# ❌ REMOVE THIS:
# model.to(device)

# ========================
# METRICS (SAFE VERSION)
# ========================
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    
    logits = predictions[0]

    intent_preds = logits["intent"].argmax(axis=1)
    manipulation_preds = logits["manipulation"].argmax(axis=1)
    request_type_preds = logits["request_type"].argmax(axis=1)
    impersonation_preds = logits["impersonation"].argmax(axis=1)

    return {
        "intent_accuracy": accuracy_score(labels["intent"], intent_preds),
        "intent_f1": f1_score(labels["intent"], intent_preds, average="weighted", zero_division=0),
        "manipulation_accuracy": accuracy_score(labels["manipulation"], manipulation_preds),
        "request_type_accuracy": accuracy_score(labels["request_type"], request_type_preds),
        "impersonation_accuracy": accuracy_score(labels["impersonation"], impersonation_preds),
    }

loading
  channel                                               text  label
0   email  We detected malware activity on your workstati...      1
1     sms  Critical virus detected on device. Pay [Amount...      1
2   email  Our security team identified your IP participa...      1
3     sms  Ransomware installed. Send [Amount] in BTC to ...      1
4   email  Your website has been injected with remote acc...      1
Dataset size: 20979


Map: 100%|██████████| 20979/20979 [00:22<00:00, 938.38 examples/s] 


In [13]:
import torch
from transformers import DataCollatorWithPadding, Trainer

# ========================
# CUSTOM DATA COLLATOR
# ========================
class MultiTaskDataCollator:
    def __init__(self, tokenizer):
        self.data_collator = DataCollatorWithPadding(tokenizer, return_tensors="pt")
    
    def __call__(self, batch):
        labels = {
            "intent": [],
            "manipulation": [],
            "request_type": [],
            "impersonation": []
        }

        input_features = []

        for item in batch:
            # ✅ Do NOT mutate original dataset item
            item_copy = item.copy()

            labels["intent"].append(item_copy.pop("intent"))
            labels["manipulation"].append(item_copy.pop("manipulation"))
            labels["request_type"].append(item_copy.pop("request_type"))
            labels["impersonation"].append(item_copy.pop("impersonation"))

            input_features.append(item_copy)

        batch_data = self.data_collator(input_features)

        # Convert labels to tensors
        for key in labels:
            batch_data[key] = torch.tensor(labels[key], dtype=torch.long)

        return batch_data


# ========================
# CUSTOM TRAINER
# ========================
class MultiTaskTrainer(Trainer):

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Extract labels
        intent_labels = inputs.pop("intent")
        manipulation_labels = inputs.pop("manipulation")
        request_type_labels = inputs.pop("request_type")
        impersonation_labels = inputs.pop("impersonation")

        # Forward pass
        outputs = model(**inputs)

        # Loss
        loss_fn = torch.nn.CrossEntropyLoss()
        loss = (
            loss_fn(outputs["intent"], intent_labels) +
            loss_fn(outputs["manipulation"], manipulation_labels) +
            loss_fn(outputs["request_type"], request_type_labels) +
            loss_fn(outputs["impersonation"], impersonation_labels)
        ) / 4.0

        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):

        # Extract labels
        labels = {
            "intent": inputs.pop("intent"),
            "manipulation": inputs.pop("manipulation"),
            "request_type": inputs.pop("request_type"),
            "impersonation": inputs.pop("impersonation"),
        }

        # Disable gradients
        with torch.no_grad():
            outputs = model(**inputs)

        loss = outputs.get("loss")

        # 🔥 Move loss to CPU (important)
        if loss is not None:
            loss = loss.detach().cpu()

        # 🔥 Respect Trainer setting
        if prediction_loss_only:
            return loss, None, None

        # 🔥 Move logits to CPU immediately (prevents OOM)
        logits = {
            k: v.detach().cpu()
            for k, v in outputs.items() if k != "loss"
        }

        # 🔥 Move labels to CPU
        labels = {
            k: v.detach().cpu()
            for k, v in labels.items()
        }

        return loss, logits, labels


# ========================
# INIT COLLATOR
# ========================
data_collator = MultiTaskDataCollator(tokenizer)

In [5]:
# Debug: Check dataset columns
print("Train dataset columns:", train_dataset.column_names)
print("Train dataset sample:")
print(train_dataset[0])

Train dataset columns: ['intent', 'manipulation', 'request_type', 'impersonation', 'input_ids', 'attention_mask']
Train dataset sample:
{'intent': 0, 'manipulation': 0, 'request_type': 0, 'impersonation': 0, 'input_ids': [101, 10373, 1024, 1000, 2128, 1024, 1031, 6335, 15916, 1033, 10514, 3366, 1022, 23999, 1029, 1006, 11689, 2904, 3621, 1007, 4931, 1010, 21358, 4886, 2243, 2009, 3475, 1005, 1056, 2524, 2012, 2035, 2000, 3443, 2115, 2219, 6310, 2544, 1997, 1996, 2417, 12707, 4487, 3367, 3217, 1012, 2045, 1005, 1055, 2130, 1037, 16985, 7384, 1031, 1015, 1033, 2007, 2023, 4919, 1999, 2568, 1012, 1996, 2111, 2040, 2328, 1996, 8418, 20924, 2544, 1997, 2417, 12707, 2109, 1037, 2291, 2714, 2000, 2056, 16985, 7384, 1012, 2311, 1037, 4487, 3367, 3217, 2029, 3594, 1037, 1008, 18667, 2094, 8831, 2291, 2738, 2084, 11575, 1010, 2139, 2497, 2030, 3649, 2052, 4012, 24759, 24695, 2477, 1037, 18819, 2062, 10047, 6806, 1012, 2009, 1008, 2003, 1008, 1037, 2204, 2801, 2295, 1012, 1045, 1005, 2310, 2042, 

In [12]:
training_args = TrainingArguments(
    output_dir="./distilbert-phishing-model",

    # 🔥 CORE
    per_device_train_batch_size=4,   # safer for 6GB GPU
    per_device_eval_batch_size=2,    # VERY IMPORTANT (prevents OOM)

    num_train_epochs=2,

    # 🔥 GPU SETTINGS
    fp16=False,  # keep False for stability on your setup

    # 🔥 MEMORY CONTROL
    eval_accumulation_steps=1,       # prevents GPU buildup
    dataloader_pin_memory=True,

    # 🔥 TRAINER BEHAVIOR
    remove_unused_columns=False,    # REQUIRED for your custom labels

    # 🔥 SAVE / LOGGING
    save_strategy="no",
    logging_strategy="steps",
    logging_steps=50,

    # 🔥 EVAL
    # evaluation_strategy="epoch",
    do_eval=True,
    # ❌ REMOVE THIS:
    # use_cpu=True
)

In [21]:
# ========================
# INSTANTIATE TRAINER
# ========================
trainer = MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Starting training...")
trainer.train()
print("Training complete!")
# ========================
# SAFE EVALUATION
# ========================
import torch
import gc

try:
    print("Preparing for evaluation...")

    # 🔥 Clean GPU completely
    gc.collect()
    torch.cuda.empty_cache()

    # 🔥 Disable cache (important for transformers)
    model.config.use_cache = False

    print("Starting evaluation...")

    results = trainer.evaluate(
        eval_dataset=val_dataset,
        metric_key_prefix="eval"
    )

    print("\nEvaluation results:")
    for k, v in results.items():
        print(f"{k}: {v}")

except RuntimeError as e:
    print("\n⚠️ CUDA issue detected:", str(e))

    print("\n🔄 Retrying with ultra-safe settings...")

    # 🔥 fallback: smallest possible batch
    trainer.args.per_device_eval_batch_size = 1

    gc.collect()
    torch.cuda.empty_cache()

    results = trainer.evaluate(
        eval_dataset=val_dataset,
        metric_key_prefix="eval"
    )

    print("\nRecovered Evaluation results:")
    for k, v in results.items():
        print(f"{k}: {v}")

except Exception as e:
    print("\n⚠️ Evaluation completed but callback failed:", str(e))

    # fallback logs
    if hasattr(trainer, 'state') and hasattr(trainer.state, 'log_history'):
        last_log = trainer.state.log_history[-1] if trainer.state.log_history else {}
        print("Last log:", last_log)

Starting training...


Step,Training Loss
50,0.758370
100,0.611720


KeyboardInterrupt: 

In [14]:
model.save_pretrained("./distilbert-phishing-model")
tokenizer.save_pretrained("./distilbert-phishing-model")

# 🔥 Save class info manually (important for custom model)
model.config.architectures = ["DistilBertMultiTaskClassifier"]
model.config.save_pretrained("./distilbert-phishing-model")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.19it/s]


In [16]:
import torch
import gc
from safetensors.torch import load_file
from transformers import AutoConfig

# ========================
# CLEAN MEMORY FIRST
# ========================
gc.collect()
torch.cuda.empty_cache()

# ========================
# LOAD MODEL (CPU FIRST)
# ========================
config = AutoConfig.from_pretrained("distilbert-base-uncased")
model = DistilBertMultiTaskClassifier(config)

# Load weights safely on CPU
state_dict = load_file("./distilbert-phishing-model/model.safetensors")
model.load_state_dict(state_dict)

model.eval()

# ========================
# MOVE TO GPU (SAFE)
# ========================
if torch.cuda.is_available():
    try:
        print("Moving model to GPU...")
        model = model.to("cuda")
    except RuntimeError as e:
        print("⚠️ GPU memory issue, staying on CPU:", e)
        model = model.to("cpu")

Moving model to GPU...


# load back

In [ ]:
import torch.nn.functional as F

def compute_phishing_score_probs(logits):
    """
    logits: dict with keys:
    intent, manipulation, request_type, impersonation
    """

    score = 0

    # Convert logits → probabilities
    manipulation_prob = F.softmax(logits["manipulation"], dim=-1)[1]
    request_probs = F.softmax(logits["request_type"], dim=-1)
    impersonation_prob = F.softmax(logits["impersonation"], dim=-1)[1]

    # Weighted scoring
    score += 2 * manipulation_prob

    score += 3 * request_probs[1]   # credentials
    score += 2 * request_probs[2]   # payment
    score += 1 * request_probs[3]   # personal info

    score += 2 * impersonation_prob

    return score.item()


def get_final_label(score, threshold=3.5):
    return 1 if score >= 4.5 else 0

In [18]:
import torch
import gc
from transformers import AutoTokenizer, AutoConfig
from safetensors.torch import load_file

model_path = "./distilbert-phishing-model"


# ========================
# CLEAN MEMORY
# ========================
gc.collect()
torch.cuda.empty_cache()

# ========================
# LOAD TOKENIZER + CONFIG
# ========================
tokenizer = AutoTokenizer.from_pretrained(model_path)
config = AutoConfig.from_pretrained(model_path)

# ========================
# INIT MODEL (CPU FIRST)
# ========================
model = DistilBertMultiTaskClassifier(config)

# Load weights safely
state_dict = load_file(f"{model_path}/model.safetensors")
model.load_state_dict(state_dict)

model.eval()

# ========================
# OPTIONAL: MOVE TO GPU
# ========================
if torch.cuda.is_available():
    try:
        print("Using GPU")
        model = model.to("cuda")
    except RuntimeError as e:
        print("⚠️ GPU not available, using CPU:", e)

Using GPU


In [19]:
def forward(
    self,
    input_ids,
    attention_mask=None,
    labels=None,
    **kwargs
):
    # Handle Trainer-style labels
    if labels is None and 'intent' in kwargs:
        labels = {
            'intent': kwargs.pop('intent'),
            'manipulation': kwargs.pop('manipulation'),
            'request_type': kwargs.pop('request_type'),
            'impersonation': kwargs.pop('impersonation'),
        }

    outputs = self.distilbert(
        input_ids=input_ids,
        attention_mask=attention_mask,
        return_dict=False  # 🔥 memory optimization
    )

    pooled_output = outputs[0][:, 0]
    pooled_output = self.dropout(pooled_output)

    intent_logits = self.intent_classifier(pooled_output)
    manipulation_logits = self.manipulation_classifier(pooled_output)
    request_type_logits = self.request_type_classifier(pooled_output)
    impersonation_logits = self.impersonation_classifier(pooled_output)

    loss = None
    if labels is not None:
        loss_fn = nn.CrossEntropyLoss()
        loss = (
            loss_fn(intent_logits, labels["intent"]) +
            loss_fn(manipulation_logits, labels["manipulation"]) +
            loss_fn(request_type_logits, labels["request_type"]) +
            loss_fn(impersonation_logits, labels["impersonation"])
        ) / 4.0

    return {
        "loss": loss,
        "intent": intent_logits,
        "manipulation": manipulation_logits,
        "request_type": request_type_logits,
        "impersonation": impersonation_logits,
    }

In [20]:
import torch
import torch.nn.functional as F

# Label mappings
INTENT_LABELS = {0: "normal", 1: "phishing", 2: "suspicious"}
REQUEST_TYPE_LABELS = {0: "none", 1: "credentials", 2: "payment", 3: "personal_info"}
BINARY_LABELS = {0: "no", 1: "yes"}


def analyze_message(text, channel="email"):
    """
    Multi-feature phishing analysis with reasoning layer
    """

    # 🔥 Get model device dynamically (VERY IMPORTANT)
    device = next(model.parameters()).device

    # Prepare input
    combined_text = f"{channel}: {text}"
    inputs = tokenizer(
        combined_text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Inference
    with torch.no_grad():
        outputs = model(**inputs)

    # Extract logits
    intent_logits = outputs["intent"]
    manipulation_logits = outputs["manipulation"]
    request_type_logits = outputs["request_type"]
    impersonation_logits = outputs["impersonation"]

    # 🔥 Compute probabilities once
    intent_probs = F.softmax(intent_logits, dim=1)
    manipulation_probs = F.softmax(manipulation_logits, dim=1)
    request_type_probs = F.softmax(request_type_logits, dim=1)
    impersonation_probs = F.softmax(impersonation_logits, dim=1)

    # Predictions
    intent_idx = intent_probs.argmax(dim=1).item()
    manipulation_idx = manipulation_probs.argmax(dim=1).item()
    request_type_idx = request_type_probs.argmax(dim=1).item()
    impersonation_idx = impersonation_probs.argmax(dim=1).item()

    # Confidence
    intent_confidence = intent_probs[0][intent_idx].item()
    manipulation_confidence = manipulation_probs[0][manipulation_idx].item()
    request_type_confidence = request_type_probs[0][request_type_idx].item()
    impersonation_confidence = impersonation_probs[0][impersonation_idx].item()

    # ========================
    # REASONING (IMPROVED)
    # ========================
    score = (
        2 * manipulation_probs[0][1] +
        3 * request_type_probs[0][1] +
        2 * request_type_probs[0][2] +
        1 * request_type_probs[0][3] +
        2 * impersonation_probs[0][1]
    ).item()

    final_label = 1 if score >= 3.5 else 0

    return {
        "text": text[:100] + "..." if len(text) > 100 else text,
        "channel": channel,
        "intent": {
            "prediction": INTENT_LABELS[intent_idx],
            "confidence": intent_confidence
        },
        "manipulation": {
            "prediction": BINARY_LABELS[manipulation_idx],
            "confidence": manipulation_confidence
        },
        "request_type": {
            "prediction": REQUEST_TYPE_LABELS[request_type_idx],
            "confidence": request_type_confidence
        },
        "impersonation": {
            "prediction": BINARY_LABELS[impersonation_idx],
            "confidence": impersonation_confidence
        },
        "score": round(score, 2),
        "final_label": "phishing" if final_label == 1 else "legitimate"
    }


# ========================
# DEMO
# ========================
print("=" * 60)
print("PHISHING DETECTION ANALYSIS")
print("=" * 60)

test_messages = [
    ("Your balance limit has been reached", "email"),
    ("Please click on the following link to verify your identity", "sms"),
    ("Confirm your banking credentials now to avoid suspension", "email"),
]

for msg, ch in test_messages:
    result = analyze_message(msg, channel=ch)

    print(f"\nMessage: {result['text']}")
    print(f"Channel: {result['channel']}")
    print(f"Intent: {result['intent']['prediction']} (conf: {result['intent']['confidence']:.2f})")
    print(f"Manipulation: {result['manipulation']['prediction']} (conf: {result['manipulation']['confidence']:.2f})")
    print(f"Request Type: {result['request_type']['prediction']} (conf: {result['request_type']['confidence']:.2f})")
    print(f"Impersonation: {result['impersonation']['prediction']} (conf: {result['impersonation']['confidence']:.2f})")
    print(f"Risk Score: {result['score']}")
    print(f"Final Label: {result['final_label'].upper()}")
    print("-" * 60)

PHISHING DETECTION ANALYSIS

Message: Your balance limit has been reached
Channel: email
Intent: normal (conf: 0.56)
Manipulation: yes (conf: 0.61)
Request Type: none (conf: 0.51)
Impersonation: yes (conf: 0.68)
Risk Score: 3.54
Final Label: PHISHING
------------------------------------------------------------

Message: Please click on the following link to verify your identity
Channel: sms
Intent: normal (conf: 0.61)
Manipulation: yes (conf: 0.67)
Request Type: none (conf: 0.50)
Impersonation: yes (conf: 0.71)
Risk Score: 3.77
Final Label: PHISHING
------------------------------------------------------------

Message: Confirm your banking credentials now to avoid suspension
Channel: email
Intent: normal (conf: 0.58)
Manipulation: yes (conf: 0.63)
Request Type: none (conf: 0.51)
Impersonation: yes (conf: 0.74)
Risk Score: 3.68
Final Label: PHISHING
------------------------------------------------------------
